In [38]:
# pip install pandas

In [39]:
df = pd.read_csv(
    r"D:\Data Analyst\ks-projects-201612.csv\ks-projects-201612.csv",
    encoding="utf-8",
    encoding_errors="replace"
)


C:\Users\npnic\AppData\Local\Temp\ipykernel_42968\793674622.py:1: DtypeWarning: Columns (0: Unnamed: 13, 1: Unnamed: 14, 2: Unnamed: 15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [40]:
print(df.head()) #check first few rows of data

          ID                                               name   \
0  1000002330                    The Songs of Adelaide & Abullah   
1  1000004038                                     Where is Hank?   
2  1000007540  ToshiCapital Rekordz Needs Help to Complete Album   
3  1000011046  Community Film Project: The Art of Neighborhoo...   
4  1000014025                               Monarch Espresso Bar   

        category  main_category  currency             deadline   goal   \
0          Poetry     Publishing       GBP  2015-10-09 11:36:00   1000   
1  Narrative Film   Film & Video       USD  2013-02-26 00:20:50  45000   
2           Music          Music       USD  2012-04-16 04:24:11   5000   
3    Film & Video   Film & Video       USD  2015-08-29 01:00:00  19500   
4     Restaurants           Food       USD  2016-04-01 13:38:27  50000   

             launched  pledged       state  backers  country  usd pledged   \
0  2015-08-11 12:12:28        0      failed        0       GB       

In [41]:
df.dtypes

ID                  int64
name                  str
category              str
main_category         str
currency              str
deadline              str
goal                  str
launched              str
pledged               str
state                 str
backers               str
country               str
usd pledged           str
Unnamed: 13        object
Unnamed: 14        object
Unnamed: 15        object
Unnamed: 16       float64
dtype: object

Data Cleaning

In [42]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")] #Drop useless columns

In [43]:
df.dtypes #check all columns now and their types

ID                int64
name                str
category            str
main_category       str
currency            str
deadline            str
goal                str
launched            str
pledged             str
state               str
backers             str
country             str
usd pledged         str
dtype: object

In [44]:
df.columns = df.columns.str.strip() #removes hidden spaces before/after every column name to clean them easier

In [45]:
# Convert date columns
df['deadline'] = pd.to_datetime(df['deadline'], errors='coerce')
df['launched'] = pd.to_datetime(df['launched'], errors='coerce')

In [46]:
#Convert numeric columns
numeric_cols = ['goal', 'pledged', 'usd pledged', 'backers']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce') 

In [47]:
df.info() #check after fixing (DONE DATA CLEANING PART)

<class 'pandas.DataFrame'>
RangeIndex: 323750 entries, 0 to 323749
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             323750 non-null  int64         
 1   name           323746 non-null  str           
 2   category       323745 non-null  str           
 3   main_category  323750 non-null  str           
 4   currency       323750 non-null  str           
 5   deadline       323118 non-null  datetime64[us]
 6   goal           323118 non-null  float64       
 7   launched       323126 non-null  datetime64[us]
 8   pledged        323126 non-null  float64       
 9   state          323750 non-null  str           
 10  backers        323127 non-null  float64       
 11  country        323750 non-null  str           
 12  usd pledged    319337 non-null  float64       
dtypes: datetime64[us](2), float64(4), int64(1), str(6)
memory usage: 32.1 MB


Data Engineering

In [48]:
df['pct_pledged'] = df['pledged'] / df['goal'].replace(0, pd.NA) #create the pct_pledged column

In [49]:
#import sql to use sql to run 
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('projects', conn, index=False, if_exists='replace')

323750

In [50]:
query = """
SELECT *,
       CASE
         WHEN pledged * 1.0 / goal >= 1 THEN 'Fully funded'
         WHEN pledged * 1.0 / goal BETWEEN 0.75 AND 1 THEN 'Nearly funded'
         ELSE 'Not nearly funded'
       END AS funding_status
FROM projects
"""

sql_df = pd.read_sql_query(query, conn)

In [51]:
sql_df.dtypes

ID                  int64
name                  str
category              str
main_category         str
currency              str
deadline              str
goal              float64
launched              str
pledged           float64
state                 str
backers           float64
country               str
usd pledged       float64
pct_pledged       float64
funding_status        str
dtype: object

In [52]:
sql_df.to_csv(
    r"D:\Data Analyst\ks-projects-201612_cleaned.csv",
    index=False
)

After opening the file, you can still see some data types issue (that you should add into the cleaning part), can you spot them out?

Currency column or Goal column (hint)

In [53]:
sql_df = sql_df.dropna(subset=['goal'])

In [54]:
sql_df.to_csv(
    r"D:\Data Analyst\ks-projects-201612_cleaned.csv",
    index=False
)